# Demo: Calabi--Yau Orientifolds and F-theory Uplifts

This notebook demonstrates how to use the `CY_orientifold` and `F_Theory_Uplift` classes as an extension of **CYTools**.

The demo has three goals:

1. Construct an orientifold of a toric Calabi--Yau hypersurface,
2. Build its F-theory uplift, and
3. Scan a small sample of models for nef partitions.

Most practical helper routines are collected in `Uplift_functions.py`, while the two main classes are implemented in `FT_with_CY_orientifold.py`.


## Setup

The files `Uplift_functions.py` and `FT_with_CY_orientifold.py` should be available from the same working directory as this notebook. In particular, `FT_with_CY_orientifold.py` imports `Uplift_functions.py`, so the import path has to be set accordingly.


In [1]:
from cytools import fetch_polytopes, Polytope
from cytools.vector_config import VectorConfiguration

import Uplift_functions as UF
import FT_with_CY_orientifold as FT


## 1. Example model: $\mathbb{P}_{[1,1,1,1,4]}$

We start with the weighted projective model $\mathbb{P}_{[1,1,1,1,4]}$. The polytope can be entered directly by its vertices, or generated from the GLSM weights using the helper function `UF.points_from_glsm`.

This model is a trilayer model, so we put it into trilayer normal form before constructing the orientifold.


In [2]:
# Direct definition by vertices.
p = Polytope([
    [-1,  0,  0,  0],
    [ 1,  1,  0,  0],
    [ 1,  0,  1,  0],
    [ 1,  0,  0,  1],
    [ 1, -1, -1, -1],
])

# Equivalent definition from GLSM weights.
p = Polytope(UF.points_from_glsm([[1, 1, 1, 1, 4]]))

print("Is trilayer:", p.is_trilayer())

p = UF.trilayer_normal_form(p)


Is trilayer: True


## 2. Constructing the orientifold

To define an orientifold, one has to specify a $\mathbb{Z}_2$ action, encoded here by a vector $\xi$.

For trilayer models if no $\xi$ is specified, it automatically selects $\xi$ to be $\xi =v_*$, the `trilayer vertex'. Alternatively, one can explicitly compute the inequivalent $\mathbb{Z}_2$ actions from the automorphism group of the polytope.


In [3]:
# Automatic construction for a trilayer model.
O11114_trilayer = FT.CY_orientifold(p)

# Explicit list of inequivalent Z2 actions.
Z2_actions = UF.inequivalent_Z2_actions(p.automorphisms(action="left"))
print(Z2_actions)


[[0 0 0 1]
 [1 0 0 0]
 [0 1 0 0]
 [1 1 0 1]
 [1 1 0 0]]


### Accepted input data

The `CY_orientifold` class accepts several kinds of input data:

- a list of points together with a $\mathbb{Z}_2$ action,
- a `Polytope`, from which all nonzero relevant points are used together with a $\mathbb{Z}_2$ action (unsell the polytope is trilayer),
- a triangulation, which is then used as the ambient triangulation, together with $\mathbb{Z}_2$ action
- a `VectorConfiguration` together with a $\mathbb{Z}_2$ action

The following two examples construct the same type of orientifold object from a vector configuration and from a triangulated vector configuration.


In [4]:
points = p.points_not_interior_to_facets()[1:]
xi = Z2_actions[1]

O_VC = FT.CY_orientifold(VectorConfiguration(points), xi)
O_VC_triangulated = FT.CY_orientifold(VectorConfiguration(points).triangulate(), xi)


### Cartier/nef refinement and triangulation

When initializing a `CY_orientifold`, the maximal refined normal fan is constructed by setting `make_Cartier_and_nef=True`.

A triangulation is constructed only if either a triangulation is supplied initially or `triangulate_points=True` is set. A1 singularities can also be resolved during initialization by setting `resolve_A1_singularities=True`.
If the normal fan is not admissible, the uplift is constructed with the old data


In [6]:
O_tri = FT.CY_orientifold(
    p,
    make_Cartier_and_nef=True,
    triangulate_points=True,
)


## 3. Basic orientifold data

The orientifold object exposes several useful pieces of toric data, including the divisor class $D_B$, the ambient toric fan, the orbifold points, and the normal fan.


In [9]:
print("Line bundle D_B:")
print(O_tri.line_bundle())

print("Ambient toric fan:")
print(O_tri.orbifold_toric_fan())

print("Orbifold points:")
print(O_tri.points_orbifold())

print("Normal fan:")
print(O_tri.normal_fan())


Line bundle D_B:
[1 0 0 0 0 0]
Ambient toric fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #6 vectors
Orbifold points:
[[-1  0  0  0]
 [ 2 -1 -1 -3]
 [ 2  0  1  1]
 [ 2  1  0  1]
 [ 2  0  0  1]
 [ 1  0  0  0]]
Normal fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #5 vectors


## 4. F-theory uplift

An `F_Theory_Uplift` object can be initialized from the same data as a `CY_orientifold`. If an orientifold object is supplied, it is used directly.

The following cell checks whether the uplift is a nef partition, prints the required blowups, constructs the relevant polytopes, and computes the Hodge numbers.


In [10]:
F = FT.F_Theory_Uplift(O_tri)

print("Is nef partition:")
print(F.is_nef_partition())

print("Blowups:")
print(F.blowups())

print("M-lattice polytopes:")
print(F.pol_B_M(), F.pol_W_M(), F.pol_M_sum(), F.pol_M_conv())

print("N-lattice polytopes:")
print(F.pol_B_N(), F.pol_W_N(), F.pol_N_sum(), F.pol_N_conv())

print("Hodge numbers:")
print("h11 =", F.h11())
print("h31 =", F.h31())


Is nef partition:
True
Blowups:
[array([[ 1,  0,  0,  0, -1,  0],
       [ 2,  0,  0,  0,  0,  1]])]
M-lattice polytopes:
A 4-dimensional lattice polytope in ZZ^6  A 6-dimensional lattice polytope in ZZ^6  A 6-dimensional reflexive lattice polytope in ZZ^6  A 6-dimensional reflexive lattice polytope in ZZ^6 
N-lattice polytopes:
A 1-dimensional lattice polytope in ZZ^6  A 6-dimensional lattice polytope in ZZ^6  A 6-dimensional reflexive lattice polytope in ZZ^6  A 6-dimensional reflexive lattice polytope in ZZ^6 
Hodge numbers:
h11 = 2
h31 = 3878


## 5. Toric fans of the uplift

The uplift keeps track of several ambient toric fans:

- the orientifold/orbifold fan $\widetilde{V}_4$,
- the singular uplift before resolving non-Higgsable clusters, and
- the smooth uplift after the required resolutions.


In [11]:
print("Orbifold toric fan:")
print(F.orbifold_toric_fan())

print("Singular uplift toric fan:")
print(F.singular_uplift_toric_fan())

print("Smooth uplift toric fan:")
print(F.smooth_uplift_toric_fan())


Orbifold toric fan:
A fine triangulation of a 4-dimensional vector configuration consisting of #6 vectors
Singular uplift toric fan:
A fine triangulation of a 6-dimensional vector configuration consisting of #9 vectors
Smooth uplift toric fan:
A fine triangulation of a 6-dimensional vector configuration consisting of #11 vectors


## 6. Line bundles

The base and Weierstrass line bundles can be represented both in the $M$-lattice and in the $N$-lattice descriptions.


In [12]:
print("M-lattice line bundles:")
print(F.line_bundle_base_M(), F.line_bundle_weierstrass_M())

print("N-lattice line bundles:")
print(F.line_bundle_base_N(), F.line_bundle_weierstrass_N())


M-lattice line bundles:
[0 0 0 ... 1 1 1] [1 1 1 ... 0 0 0]
N-lattice line bundles:
[1 0 0 0 0 0 0 0 0 0 0] [0 1 1 1 1 1 1 1 1 1 1]


## 7. Intersection numbers

The following command computes the intersection numbers of the ambient toric sixfold of the smooth uplift.


In [13]:
print(F.intersection_numbers_smooth_uplift())


{(1, 2, 3, 4, 7, 8): 1.0, (1, 2, 3, 4, 7, 9): 0.3333333333, (1, 2, 3, 4, 8, 9): 0.5, (1, 2, 3, 5, 7, 8): 1.0, (1, 2, 3, 5, 7, 9): 0.3333333333, (1, 2, 3, 5, 8, 9): 0.5, (1, 2, 4, 5, 7, 8): 1.0, (1, 2, 4, 5, 7, 9): 0.3333333333, (1, 2, 4, 5, 8, 9): 0.5, (1, 3, 4, 5, 7, 8): 1.0, (1, 3, 4, 5, 7, 9): 0.3333333333, (1, 3, 4, 5, 8, 9): 0.5, (2, 3, 4, 6, 7, 9): 0.3333333333, (2, 3, 4, 6, 7, 11): 0.3333333333, (2, 3, 4, 6, 8, 9): 0.5, (2, 3, 4, 6, 8, 10): 1.0, (2, 3, 4, 6, 10, 11): 1.0, (2, 3, 4, 7, 8, 10): 1.0, (2, 3, 4, 7, 10, 11): 1.0, (2, 3, 5, 6, 7, 9): 0.3333333333, (2, 3, 5, 6, 7, 11): 0.3333333333, (2, 3, 5, 6, 8, 9): 0.5, (2, 3, 5, 6, 8, 10): 1.0, (2, 3, 5, 6, 10, 11): 1.0, (2, 3, 5, 7, 8, 10): 1.0, (2, 3, 5, 7, 10, 11): 1.0, (2, 4, 5, 6, 7, 9): 0.3333333333, (2, 4, 5, 6, 7, 11): 0.3333333333, (2, 4, 5, 6, 8, 9): 0.5, (2, 4, 5, 6, 8, 10): 1.0, (2, 4, 5, 6, 10, 11): 1.0, (2, 4, 5, 7, 8, 10): 1.0, (2, 4, 5, 7, 10, 11): 1.0, (3, 4, 5, 6, 7, 9): 0.3333333333, (3, 4, 5, 6, 7, 11): 0.333333

## 8. A small scan at $h^{1,1}=2$

As a final example, we scan the first 100 polytopes with $h^{1,1}=2$ and collect those orientifold actions whose F-theory uplift gives a nef partition.

This is intentionally a small demo scan. For larger scans, one should add progress logging, exception handling, and persistent output files.


In [14]:
nef_partitions = []

for polytope in fetch_polytopes(h11=2, limit=100):
    automorphisms = polytope.automorphisms(action="left")
    z2_actions = UF.inequivalent_Z2_actions(automorphisms)

    for xi in z2_actions:
        uplift = FT.F_Theory_Uplift(
            polytope,
            xi,
            make_Cartier_and_nef=True,
        )

        if uplift.is_nef_partition():
            nef_partitions.append(uplift)


### Number of nef partitions found


In [15]:
print(len(nef_partitions))


72


## 9. Delayed triangulation

For scans, it can be useful to postpone triangulation in order to save computation time. If `triangulate_points=True` is not set during initialization, the triangulation can be constructed later with `.triangulate()`.
If the uplift yields a nef decomposition, the triangulation is automatically chosen such that $D_B$ and $D_W$ are Cartier and nef in this triangulation

In [16]:
Fnef = nef_partitions[1]

print("Triangulated before call:", Fnef.triangulated())

Fnef.triangulate()

print("Triangulated after call:", Fnef.triangulated())


Triangulated before call: False
Triangulated after call: True


## Summary

This notebook illustrated the basic workflow:

1. define a toric polytope,
2. choose or compute a $\mathbb{Z}_2$ orientifold action,
3. construct a `CY_orientifold`,
4. build the corresponding `F_Theory_Uplift`, and
5. inspect nef partitions, line bundles, fans, Hodge numbers, and intersection data.
